# speed_lstm on real I-24 data

Mounts Drive, clones the code from GitHub, installs deps, and trains the 3D-mode model against `Final-Project-CHULA/data`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

candidates = [
    '/content/drive/MyDrive/Final-Project-CHULA/data',
    '/content/drive/Shareddrives/Final-Project-CHULA/data',
]
DATA_DIR = None
for c in candidates:
    if os.path.isdir(c):
        DATA_DIR = c
        break

if DATA_DIR is None:
    print('Could not auto-find the dataset. Searching MyDrive for a matching folder...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if root.count(os.sep) - '/content/drive/MyDrive'.count(os.sep) > 3:
            dirs[:] = []
            continue
        if {'obj', 'ts', 'hg'}.issubset(set(dirs)):
            DATA_DIR = root
            break

print('DATA_DIR =', DATA_DIR)
assert DATA_DIR is not None, 'Set DATA_DIR manually to the folder containing obj/, ts/, hg/'
print(os.listdir(DATA_DIR))

In [ ]:
!git clone https://github.com/WalkerCze3/capcapcar-speed.git
%cd capcapcar-speed
!pip install -q -r requirements.txt

In [ ]:
!python scripts/train_v2_cli.py --data-dir "{DATA_DIR}" --mode 3d --out runs/v2/3d --epochs 20

In [ ]:
!python scripts/train_v2_cli.py --data-dir "{DATA_DIR}" --mode 2d --out runs/v2/2d --epochs 20
!python scripts/train_v2_cli.py --data-dir "{DATA_DIR}" --mode combined --out runs/v2/combined --epochs 20

In [ ]:
import sys
sys.path.insert(0, 'src')
from speed_lstm.balanced_report import verify_consistent_test_sets
import json

report = verify_consistent_test_sets({
    '2d': 'runs/v2/2d', '3d': 'runs/v2/3d', 'combined': 'runs/v2/combined',
})
print(json.dumps(report, indent=2))